In [46]:
import pathlib
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [47]:
RAW_DIR = pathlib.Path("data/raw")
PROC_DIR = pathlib.Path("data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())

RAW_DIR: /Users/that_bat/bootcamp_moshi_wang/homework/homework0/homework06/data/raw
PROC_DIR: /Users/that_bat/bootcamp_moshi_wang/homework/homework0/homework06/data/processed


In [48]:
csv_path = RAW_DIR / "starter_data.csv"
df = pd.read_csv(csv_path)
print("shape:", df.shape)
df.head()

shape: (10, 3)


,category,value,date
0,A,10,2025-08-01
1,B,15,2025-08-02
2,A,12,2025-08-03
3,B,18,2025-08-04
4,C,25,2025-08-05


In [49]:
df["date"] = pd.to_datetime(df["date"])
print("data types:")
print(df.dtypes)

print("\nmissing")
print(df.isna().sum())

data types:
category               str
value                int64
date        datetime64[us]
dtype: object

missing
category    0
value       0
date        0
dtype: int64


In [50]:
def fill_missing_median(df, columns=None):
    df_copy = df.copy()

    if columns is None:
        columns = df_copy.select_dtypes(include=np.number).columns

    for col in columns:
        df_copy[col] = df_copy[col].fillna(df_copy[col].median())

    return df_copy

In [51]:
test_df = df.copy()
test_df.loc[2, "value"] = None
print(test_df)

  category  value       date
0        A   10.0 2025-08-01
1        B   15.0 2025-08-02
2        A    NaN 2025-08-03
3        B   18.0 2025-08-04
4        C   25.0 2025-08-05
5        C   30.0 2025-08-06
6        A   11.0 2025-08-07
7        B   14.0 2025-08-08
8        C   28.0 2025-08-09
9        A   13.0 2025-08-10


In [52]:
filled_df = fill_missing_median(test_df, ["value"])
print(filled_df)

  category  value       date
0        A   10.0 2025-08-01
1        B   15.0 2025-08-02
2        A   15.0 2025-08-03
3        B   18.0 2025-08-04
4        C   25.0 2025-08-05
5        C   30.0 2025-08-06
6        A   11.0 2025-08-07
7        B   14.0 2025-08-08
8        C   28.0 2025-08-09
9        A   13.0 2025-08-10


In [53]:
def drop_missing(df, columns=None, threshold=None):
    df_copy = df.copy()

    if columns is not None:
        return df_copy.dropna(subset=columns)

    if threshold is not None:
        return df_copy.dropna(
            thresh=int(threshold * df_copy.shape[1])
        )

    return df_copy.dropna()

In [54]:
drop_df = drop_missing(test_df, ["value"])
print(drop_df)

  category  value       date
0        A   10.0 2025-08-01
1        B   15.0 2025-08-02
3        B   18.0 2025-08-04
4        C   25.0 2025-08-05
5        C   30.0 2025-08-06
6        A   11.0 2025-08-07
7        B   14.0 2025-08-08
8        C   28.0 2025-08-09
9        A   13.0 2025-08-10


In [55]:
def normalize_data(df, columns=None, method="minmax"):
    df_copy = df.copy()

    if columns is None:
        columns = df_copy.select_dtypes(include=np.number).columns

    if method == "minmax":
        scaler = MinMaxScaler()
    else:
        scaler = StandardScaler()

    df_copy[columns] = scaler.fit_transform(df_copy[columns])

    return df_copy

In [56]:
normalized_df = normalize_data(df, ["value"])
print(normalized_df)

  category  value       date
0        A   0.00 2025-08-01
1        B   0.25 2025-08-02
2        A   0.10 2025-08-03
3        B   0.40 2025-08-04
4        C   0.75 2025-08-05
5        C   1.00 2025-08-06
6        A   0.05 2025-08-07
7        B   0.20 2025-08-08
8        C   0.90 2025-08-09
9        A   0.15 2025-08-10


In [57]:
outlier_df = df.copy()
outlier_df.loc[4, "value"] = 100
print(outlier_df)

  category  value       date
0        A     10 2025-08-01
1        B     15 2025-08-02
2        A     12 2025-08-03
3        B     18 2025-08-04
4        C    100 2025-08-05
5        C     30 2025-08-06
6        A     11 2025-08-07
7        B     14 2025-08-08
8        C     28 2025-08-09
9        A     13 2025-08-10


In [58]:
Q1 = outlier_df["value"].quantile(0.25)
Q3 = outlier_df["value"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Q1: 12.25
Q3: 25.5
IQR: 13.25
Lower bound: -7.625
Upper bound: 45.375


In [59]:
clean_df = df.copy()

clean_df["date"] = pd.to_datetime(clean_df["date"])

clean_df = fill_missing_median(clean_df)
clean_df = drop_missing(clean_df)
clean_df = normalize_data(clean_df, ["value"])

print(clean_df)

  category  value       date
0        A   0.00 2025-08-01
1        B   0.25 2025-08-02
2        A   0.10 2025-08-03
3        B   0.40 2025-08-04
4        C   0.75 2025-08-05
5        C   1.00 2025-08-06
6        A   0.05 2025-08-07
7        B   0.20 2025-08-08
8        C   0.90 2025-08-09
9        A   0.15 2025-08-10


In [60]:
print("original data")
print(df)

print("\ncleaned")
print(clean_df)

original data
  category  value       date
0        A     10 2025-08-01
1        B     15 2025-08-02
2        A     12 2025-08-03
3        B     18 2025-08-04
4        C     25 2025-08-05
5        C     30 2025-08-06
6        A     11 2025-08-07
7        B     14 2025-08-08
8        C     28 2025-08-09
9        A     13 2025-08-10

cleaned
  category  value       date
0        A   0.00 2025-08-01
1        B   0.25 2025-08-02
2        A   0.10 2025-08-03
3        B   0.40 2025-08-04
4        C   0.75 2025-08-05
5        C   1.00 2025-08-06
6        A   0.05 2025-08-07
7        B   0.20 2025-08-08
8        C   0.90 2025-08-09
9        A   0.15 2025-08-10


In [61]:
print("original shape:", df.shape)
print("cleaned", clean_df.shape)

print("\noriginal data type")
print(df.dtypes)

print("\ncleaned")
print(clean_df.dtypes)

print("\nmissing value after cleaning:")
print(clean_df.isna().sum())

original shape: (10, 3)
cleaned (10, 3)

original data type
category               str
value                int64
date        datetime64[us]
dtype: object

cleaned
category               str
value              float64
date        datetime64[us]
dtype: object

missing value after cleaning:
category    0
value       0
date        0
dtype: int64


In [63]:
output_path = PROC_DIR / "cleaned_data.csv"

clean_df.to_csv(output_path, index=False)

print("cleaned data saved in", output_path)

cleaned data saved in data/processed/cleaned_data.csv


In [65]:
from src.cleaning import fill_missing_median, drop_missing, normalize_data
print(fill_missing_median.__module__)
print(drop_missing.__module__)
print(normalize_data.__module__)

src.cleaning
src.cleaning
src.cleaning


In [66]:
final_df = df.copy()
final_df = fill_missing_median(final_df)
final_df = drop_missing(final_df, threshold=0.5)
final_df = normalize_data(final_df)

print(final_df)

  category  value       date
0        A   0.00 2025-08-01
1        B   0.25 2025-08-02
2        A   0.10 2025-08-03
3        B   0.40 2025-08-04
4        C   0.75 2025-08-05
5        C   1.00 2025-08-06
6        A   0.05 2025-08-07
7        B   0.20 2025-08-08
8        C   0.90 2025-08-09
9        A   0.15 2025-08-10


In [67]:
print("final shape", final_df.shape)

print("\ndata type")
print(final_df.dtypes)

print("\nmissing")
print(final_df.isna().sum())

print("\nvalue range")
print("min:", final_df["value"].min())
print("max:", final_df["value"].max())

final shape (10, 3)

data type
category               str
value              float64
date        datetime64[us]
dtype: object

missing
category    0
value       0
date        0
dtype: int64

value range
min: 0.0
max: 1.0


In [68]:
output_path = PROC_DIR / "cleaned_data.csv"
final_df.to_csv(output_path, index=False)
print("cleaned data saved in", output_path)

cleaned data saved in data/processed/cleaned_data.csv
